In [1]:
import os

os.makedirs("outputs", exist_ok=True)
os.makedirs("excel", exist_ok=True)

generate_data_code = '''import numpy as np
import pandas as pd


def generate_duka_sales(output_path: str = "duka_sales.csv") -> pd.DataFrame:
    """Generate synthetic Duka Electronics H1 2026 sales data (seed 2026)."""
    rng = np.random.default_rng(2026)
    n = 1000
    products = {
        "Phone": 15000,
        "Laptop": 65000,
        "Headphones": 2500,
        "Tablet": 28000,
        "Charger": 1200,
    }

    sales = pd.DataFrame({
        "order_id": range(1001, 1001 + n),
        "order_date": pd.to_datetime("2026-01-01")
        + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
        "branch": rng.choice(
            ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
            n,
            p=[0.4, 0.2, 0.15, 0.15, 0.1],
        ),
        "product": rng.choice(list(products), n),
        "quantity": rng.integers(1, 6, n),
        "channel": rng.choice(
            ["In-store", "Online", "Phone order"],
            n,
            p=[0.55, 0.35, 0.10],
        ),
    })
    sales["unit_price"] = sales["product"].map(products)
    sales.to_csv(output_path, index=False)
    return sales


def generate_agent_network(
    agents_path: str = "agents.csv",
    txns_path: str = "agent_transactions.csv",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Generate synthetic mobile money agents and transaction extract (seed 44)."""
    rng = np.random.default_rng(44)
    n = 600
    agents = pd.DataFrame({
        "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
        "county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),
        "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS").strftime("%Y-%m-%d"),
    })
    txns = pd.DataFrame({
        "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
        "agent_id": rng.choice(agents["agent_id"], n),
        "txn_type": rng.choice(
            ["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"], n
        ),
        "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
        "txn_date": (
            pd.to_datetime("2026-03-01")
            + pd.to_timedelta(rng.integers(0, 60, n), unit="D")
        ).strftime("%Y-%m-%d"),
    })
    idx = rng.choice(n, 30, replace=False)
    txns.loc[idx[:15], "amount"] = np.nan
    txns.loc[idx[15:20], "amount"] = "N/A"
    txns.loc[idx[20:25], "amount"] = "250000"
    txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")
    txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)
    txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"
    agents.to_csv(agents_path, index=False)
    txns.to_csv(txns_path, index=False)
    return agents, txns


if __name__ == "__main__":
    generate_duka_sales()
    generate_agent_network()
'''

with open("generate_data.py", "w") as f:
    f.write(generate_data_code.strip() + "\n")

!python generate_data.py
!ls -la

total 108
drwxr-xr-x 1 root root  4096 Nov 24 05:13 .
drwxr-xr-x 1 root root  4096 Nov 24 05:02 ..
-rw-r--r-- 1 root root   517 Nov 24 05:16 agents.csv
-rw-r--r-- 1 root root 25094 Nov 24 05:16 agent_transactions.csv
drwxr-xr-x 4 root root  4096 Nov 18 15:58 .config
-rw-r--r-- 1 root root 47772 Nov 24 05:16 duka_sales.csv
drwxr-xr-x 2 root root  4096 Nov 24 05:13 excel
-rw-r--r-- 1 root root  2775 Nov 24 05:16 generate_data.py
drwxr-xr-x 2 root root  4096 Nov 24 05:13 outputs
drwxr-xr-x 1 root root  4096 Nov 18 15:58 sample_data


# Duka Electronics — H1 2026 Sales Exploration

In [2]:
import numpy as np
import pandas as pd

sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("Shape:", sales.shape)
print("\nData types:\n", sales.dtypes)
print("\nMissing values:\n", sales.isna().sum())
print("\nDate range:", sales["order_date"].min().date(), "to", sales["order_date"].max().date())
print("\nBranch counts:\n", sales["branch"].value_counts())
print("\nProduct counts:\n", sales["product"].value_counts())
print("\nChannel counts:\n", sales["channel"].value_counts())
sales.head()

Shape: (1000, 7)

Data types:
 order_id               int64
order_date    datetime64[ns]
branch                object
product               object
quantity               int64
channel               object
unit_price             int64
dtype: object

Missing values:
 order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

Date range: 2026-01-01 to 2026-06-29

Branch counts:
 branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117
Name: count, dtype: int64

Product counts:
 product
Laptop        220
Tablet        215
Phone         194
Headphones    187
Charger       184
Name: count, dtype: int64

Channel counts:
 channel
In-store       579
Online         320
Phone order    101
Name: count, dtype: int64


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


## Executive Summary
* Duka Electronics generated **KES 71,792,500** in total revenue across **1,000** orders from January 1, 2026 to June 29, 2026.
* The Nairobi branch led performance with **KES 26,106,100** (**36.36%** of total revenue across **376** orders), followed by Mombasa at **KES 14,003,700**.
* Laptops dominated product revenue at **KES 41,535,000** (**57.85%** of revenue), whereas Tablets ranked first in unit volume with **691** units sold.
* April 2026 (`2026-04`) was the strongest month at **KES 13,518,300**, outperforming June 2026 (`2026-06`, the lowest month at **KES 10,903,400**) by **23.98%**.
* Premium online orders from Nairobi and Mombasa totaled **83** orders and **KES 10,218,000** in revenue.

In [3]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.to_period("M").astype(str)
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5
sales["price_band"] = np.where(sales["unit_price"] >= 28000, "Premium", "Value")

sales[[
    "order_id", "order_date", "branch", "product", "quantity",
    "unit_price", "revenue", "month", "weekday", "is_weekend", "price_band"
]].head()

,order_id,order_date,branch,product,quantity,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,15000,45000,2026-06,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,65000,260000,2026-02,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,28000,140000,2026-01,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,2500,12500,2026-04,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,2500,7500,2026-03,Saturday,True,Value


## 1. Data Loading & First-Five-Minutes Inspection
Loading `duka_sales.csv` with `parse_dates=["order_date"]` to verify shape, column types, missing values, date range, and key categories.

In [4]:
branch_perf = (
    sales.groupby("branch")
    .agg(
        total_revenue=("revenue", "sum"),
        order_count=("order_id", "count"),
        avg_order_revenue=("revenue", "mean"),
    )
    .sort_values("total_revenue", ascending=False)
)
branch_perf["revenue_share_pct"] = (branch_perf["total_revenue"] / sales["revenue"].sum() * 100).round(2)
branch_perf["revenue_rank"] = branch_perf["total_revenue"].rank(ascending=False).astype(int)
branch_perf["order_rank"] = branch_perf["order_count"].rank(ascending=False).astype(int)
branch_perf["avg_order_revenue"] = branch_perf["avg_order_revenue"].round(2)
branch_perf

,total_revenue,order_count,avg_order_revenue,revenue_share_pct,revenue_rank,order_rank
branch,,,,,,
Nairobi,26106100,376,69431.12,36.36,1,1
Mombasa,14003700,186,75288.71,19.51,2,2
Nakuru,11970400,162,73891.36,16.67,3,3
Kisumu,10868100,159,68352.83,15.14,4,4
Eldoret,8844200,117,75591.45,12.32,5,5


### First-Five-Minutes Inspection Findings
* **Shape:** `1,000` rows and `7` columns (`order_id`, `order_date`, `branch`, `product`, `quantity`, `channel`, `unit_price`).
* **Types & Missingness:** `order_date` is parsed as `datetime64[ns]`; numeric and text columns match expected types, with **0** missing values in every column.
* **Date Range:** Runs for **180** days from **2026-01-01** to **2026-06-29**.
* **Key Categories:** `5` branches (`Nairobi`, `Mombasa`, `Kisumu`, `Nakuru`, `Eldoret`), `5` products (`Phone`, `Laptop`, `Headphones`, `Tablet`, `Charger`), and `3` channels (`In-store`, `Online`, `Phone order`).

In [5]:
prod_perf = sales.groupby("product").agg(
    total_revenue=("revenue", "sum"),
    units_sold=("quantity", "sum"),
    unit_price=("unit_price", "first"),
)

top3_revenue = prod_perf.sort_values("total_revenue", ascending=False).head(3)
top3_units = prod_perf.sort_values("units_sold", ascending=False).head(3)

print("Top 3 Products by Revenue:")
display(top3_revenue[["total_revenue", "units_sold", "unit_price"]])

print("\nTop 3 Products by Units Sold:")
display(top3_units[["units_sold", "total_revenue", "unit_price"]])

Top 3 Products by Revenue:


,total_revenue,units_sold,unit_price
product,,,
Laptop,41535000,639,65000
Tablet,19348000,691,28000
Phone,8880000,592,15000



Top 3 Products by Units Sold:


,units_sold,total_revenue,unit_price
product,,,
Tablet,691,19348000,28000
Laptop,639,41535000,65000
Phone,592,8880000,15000


## 2. Derived Columns (`revenue`, `month`, `weekday`, `is_weekend`, `price_band`)
Creating `revenue`, `month`, `weekday`, `is_weekend`, and `price_band` (`Premium` for `unit_price >= 28000`; `Value` otherwise).

In [6]:
wknd_perf = sales.groupby("is_weekend").agg(
    orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    avg_order_revenue=("revenue", "mean"),
)
wknd_perf.index = wknd_perf.index.map({False: "Weekday", True: "Weekend"})
wknd_perf["order_share_pct"] = (wknd_perf["orders"] / len(sales) * 100).round(2)
wknd_perf["revenue_share_pct"] = (wknd_perf["total_revenue"] / sales["revenue"].sum() * 100).round(2)
wknd_perf["avg_order_revenue"] = wknd_perf["avg_order_revenue"].round(2)
wknd_perf

,orders,total_revenue,avg_order_revenue,order_share_pct,revenue_share_pct
is_weekend,,,,,
Weekday,723,51856200,71723.65,72.3,72.23
Weekend,277,19936300,71972.20,27.7,27.77


## Task 3: Branch Rankings by Revenue and Order Count

In [7]:
monthly_perf = (
    sales.groupby("month")
    .agg(total_revenue=("revenue", "sum"), orders=("order_id", "count"))
    .sort_values("total_revenue", ascending=False)
)

best_month = monthly_perf.index[0]
worst_month = monthly_perf.index[-1]
best_rev = int(monthly_perf.loc[best_month, "total_revenue"])
worst_rev = int(monthly_perf.loc[worst_month, "total_revenue"])

pct_above_worst = (best_rev - worst_rev) / worst_rev * 100
pct_diff_midpoint = (best_rev - worst_rev) / ((best_rev + worst_rev) / 2) * 100

display(monthly_perf)
print(f"Best month: {best_month} (KES {best_rev:,})")
print(f"Worst month: {worst_month} (KES {worst_rev:,})")
print(f"Difference: KES {best_rev - worst_rev:,} | {pct_above_worst:.2f}% above worst | {pct_diff_midpoint:.2f}% relative to average")

,total_revenue,orders
month,,
2026-04,13518300,167
2026-03,12396800,186
2026-01,11786600,173
2026-02,11772000,159
2026-05,11415400,153
2026-06,10903400,162


Best month: 2026-04 (KES 13,518,300)
Worst month: 2026-06 (KES 10,903,400)
Difference: KES 2,614,900 | 23.98% above worst | 21.41% relative to average


**Answer (Task 3):** Yes—the revenue ranking and order-count ranking are identical from 1st to 5th: **Nairobi** (KES 26,106,100; 376 orders; 36.36% share), **Mombasa** (KES 14,003,700; 186 orders; 19.51% share), **Nakuru** (KES 11,970,400; 162 orders; 16.67% share), **Kisumu** (KES 10,868,100; 159 orders; 15.14% share), and **Eldoret** (KES 8,844,200; 117 orders; 12.32% share). Although average order revenue differs slightly across branches (lowest in Kisumu at KES 68,352.83 and highest in Eldoret at KES 75,591.45), order volume differences are large enough that no branch overtakes another in total revenue.

In [8]:
premium_coast_cap = sales[
    (sales["price_band"] == "Premium")
    & (sales["channel"] == "Online")
    & (sales["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

print("Order count:", len(premium_coast_cap))
print("Total revenue (KES):", int(premium_coast_cap["revenue"].sum()))

premium_coast_cap.to_csv("outputs/premium_online_coast_capital.csv", index=False)
premium_coast_cap.head()

Order count: 83
Total revenue (KES): 10218000


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
6,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,2026-01,Thursday,False,Premium
13,1014,2026-06-12,Nairobi,Tablet,3,Online,28000,84000,2026-06,Friday,False,Premium
32,1033,2026-04-29,Nairobi,Laptop,2,Online,65000,130000,2026-04,Wednesday,False,Premium
35,1036,2026-03-02,Nairobi,Laptop,4,Online,65000,260000,2026-03,Monday,False,Premium
39,1040,2026-02-10,Mombasa,Laptop,5,Online,65000,325000,2026-02,Tuesday,False,Premium


## Task 4: Top Three Products by Revenue and Units Sold

**Answer (Task 4):** Top three by **revenue** are **Laptop** (KES 41,535,000; 639 units), **Tablet** (KES 19,348,000; 691 units), and **Phone** (KES 8,880,000; 592 units). Top three by **units sold** are **Tablet** (691 units), **Laptop** (639 units), and **Phone** (592 units). They contain the same three products in a different top-two order because a Laptop costs **KES 65,000** versus **KES 28,000** for a Tablet—so fewer Laptop units generate more than double the revenue.

## Task 5: Weekend vs. Weekday Orders Comparison

**Answer (Task 5):** Weekdays record **723** orders (**72.30%**), **KES 51,856,200** (**72.23%** of revenue), and **KES 71,723.65** average order revenue. Weekends record **277** orders (**27.70%**), **KES 19,936,300** (**27.77%** of revenue), and **KES 71,972.20** average order revenue. Order volume and total revenue are higher on weekdays simply because there are five weekdays per week versus two weekend days, while average order revenue is virtually tied (weekend is higher by KES 248.55, or +0.35%).

## Task 6: Best Month vs. Worst Month Revenue

**Answer (Task 6):** The best month is **2026-04** (April) with **KES 13,518,300**, and the worst month is **2026-06** (June) with **KES 10,903,400**—a difference of **KES 2,614,900**. April's revenue is **23.98%** higher than June's revenue (**21.41%** relative difference using their average).

## Task 7: Premium Online Orders in Nairobi and Mombasa

**Answer (Task 7):** There are **83** Premium online orders from Nairobi and Mombasa, generating **KES 10,218,000** in total revenue. The filtered dataset has been saved to `outputs/premium_online_coast_capital.csv` with `index=False`.

In [9]:
import os
from google.colab import files

os.makedirs("excel", exist_ok=True)
uploaded = files.upload()  # Select duka_branch_summary.xlsx from your computer

# Move it to excel/duka_branch_summary.xlsx
for filename in uploaded.keys():
    os.replace(filename, "excel/duka_branch_summary.xlsx")

print("Exists:", os.path.exists("excel/duka_branch_summary.xlsx"))

Saving excel_duka_branch_summary.xlsx to excel_duka_branch_summary.xlsx
Exists: True
